In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models,datasets,layers
from tensorflow.keras.models import Sequential

In [ ]:
import cv2
import os
import PIL.Image as Image
import tensorflow_hub as hub

In [ ]:
import pathlib
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

data_dir = tf.keras.utils.get_file(
    'flower_photos',
    origin=dataset_url,
    untar=True
)

print(data_dir)

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
/root/.keras/datasets/flower_photos


In [ ]:
img_height=224
img_width=224
batch_size=32

train_ds=tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height,img_width),
    batch_size=batch_size
)

val_ds=tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height,img_width),
    batch_size=batch_size
)
class_names=train_ds.class_names
print(class_names)

Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
['flower_photos']


Optimize Dataset

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

train_ds=train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds=val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Building CNN model before premodel

In [ ]:
cnn_model=models.Sequential([
    layers.Rescaling(1./255,input_shape=(224,224,3)),

    layers.Conv2D(32,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(5,activation='softmax')

])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history=cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


92/92 ━━━━━━━━━━━━━━━━━━━━ 358s 4s/step - accuracy: 0.9891 - loss: 0.0171 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 418s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 377s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 337s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 363s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00


Lets use Transfer Learning (MobileNetV2)

In [ ]:
base_model=tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable=False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Building a model using pre trained model

In [ ]:
transfer_model=models.Sequential([
    layers.Rescaling(1./225),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dense(5,activation='softmax')
])

transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
transfer_history=transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 156s 2s/step - accuracy: 0.9946 - loss: 0.0151 - val_accuracy: 1.0000 - val_loss: 5.1319e-07
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 1.0000 - loss: 2.5137e-07 - val_accuracy: 1.0000 - val_loss: 5.0977e-07
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 150s 2s/step - accuracy: 1.0000 - loss: 2.4970e-07 - val_accuracy: 1.0000 - val_loss: 5.0555e-07
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 150s 2s/step - accuracy: 1.0000 - loss: 2.4743e-07 - val_accuracy: 1.0000 - val_loss: 5.0182e-07
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 1.0000 - loss: 2.4536e-07 - val_accuracy: 1.0000 - val_loss: 4.9662e-07


Lets compare and see which model is giving more accuracy and how fast they are worjing and the use of transfer learning

In [ ]:
print("CNN final accuracy:",cnn_history.history['val_accuracy'][-1])
print("Transfer learning accuracy:",transfer_history.history['val_accuracy'][-1])

CNN final accuracy: 1.0
Transfer learning accuracy: 1.0
